# Subgroup SHAP Analysis

This notebook analyses how feature importance differs across protected groups (Male vs Female).

It evaluates:
- Baseline model behaviour
- Exponentiated Gradient (DP)

The goal is to assess whether fairness mitigation reduces disparities in model reasoning, not just outcomes.

In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.data.loader import load_hr_analytics
from src.models.baseline import get_logistic_regression_pipeline

# Load data
ds = load_hr_analytics("../data/raw/HR_Analytics.csv")

out = Path("../runs/shap_subgroups")
out.mkdir(parents=True, exist_ok=True)





## Baseline Model

This section computes SHAP values for the baseline logistic regression model and evaluates subgroup differences in feature importance.

Subgroup analysis is performed by comparing mean absolute SHAP values across Male and Female groups.

In [6]:
# ================================
# BASELINE MODEL — SHAP ANALYSIS
# ================================

# Train baseline model
pipeline = get_logistic_regression_pipeline(ds.X)
pipeline.fit(ds.X, ds.y)

# Extract preprocessing + model
preprocessor = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]

# Encode features
X_processed = preprocessor.transform(ds.X).toarray()

# Compute SHAP values (baseline)
explainer_baseline = shap.Explainer(
    model,
    X_processed,
    feature_names=preprocessor.get_feature_names_out()
)

shap_vals_baseline = explainer_baseline(X_processed)

# Prepare variables for subgroup analysis
shap_values = shap_vals_baseline
X_enc = X_processed
A = ds.A.reset_index(drop=True)  # Protected attribute (Gender)
feature_names = preprocessor.get_feature_names_out()


### Subgroup Feature Importance (Baseline)

The following plots show:
- Mean SHAP importance for top features across groups
- Differences in feature influence between Male and Female groups

This helps identify whether the model relies differently on features for different groups.

In [7]:

# ================================
# SUBGROUP SPLIT (BASELINE)
# ================================

male_idx = A[A == "Male"].index
female_idx = A[A == "Female"].index

shap_male = shap_values.values[male_idx]
shap_female = shap_values.values[female_idx]

# Mean absolute SHAP importance per group
male_importance = np.abs(shap_male).mean(axis=0)
female_importance = np.abs(shap_female).mean(axis=0)

# Combine into DataFrame
df_imp = pd.DataFrame({
    "feature": feature_names,
    "Male": male_importance,
    "Female": female_importance,
})

# Difference (Male − Female)
df_imp["diff"] = df_imp["Male"] - df_imp["Female"]


# ================================
# OUTPUT DIRECTORY
# ================================

out = Path("../runs/shap_subgroups")
out.mkdir(parents=True, exist_ok=True)


# ================================
# PLOT 1 — FEATURE IMPORTANCE
# ================================

top = df_imp.sort_values("Male", ascending=False).head(5)

x = np.arange(len(top))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, top["Male"], width, label="Male")
plt.bar(x + width/2, top["Female"], width, label="Female")

plt.xticks(x, top["feature"], rotation=45, ha="right")
plt.ylabel("Mean |SHAP|")
plt.title("Subgroup Feature Importance (Baseline)")
plt.legend()

plt.tight_layout()
plt.savefig(out / "shap_subgroup_baseline.png", dpi=300)
plt.close()


# ================================
# PLOT 2 — SHAP DIFFERENCES
# ================================

top_diff = df_imp.sort_values("diff", key=abs, ascending=False).head(5)

plt.figure(figsize=(7, 4))
plt.barh(top_diff["feature"], top_diff["diff"])
plt.axvline(0, color="black", linewidth=0.8)

plt.xlabel("Male − Female Mean |SHAP|")
plt.title("Subgroup SHAP Differences (Baseline)")

plt.tight_layout()
plt.savefig(out / "shap_subgroup_diff_baseline.png", dpi=300)
plt.close()



## Exponentiated Gradient (DP)

This section repeats the subgroup SHAP analysis after applying in-processing mitigation using Exponentiated Gradient with a Demographic Parity constraint.

The aim is to assess whether mitigation reduces disparities in feature attribution between groups.

In [8]:

# ============================================
# EXPONENTIATED GRADIENT (DP) — SHAP ANALYSIS
# ============================================

from src.mitigation.exponentiated_gradient import train_expgrad
from src.data.preprocessing import build_preprocessor

# Preprocess for EG model
pre_eg = build_preprocessor(ds.X)
X_enc_eg = pre_eg.fit_transform(ds.X).toarray()

# Train EG model (DP constraint)
eg_dp = train_expgrad(
    X_enc_eg,
    ds.y,
    ds.A,
    constraint="dp",
    seed=42,
)

eg_predictor = eg_dp.predictors_[0]

# Compute SHAP values
explainer_eg = shap.Explainer(
    eg_predictor,
    X_enc_eg,
    feature_names=pre_eg.get_feature_names_out()
)

shap_vals_eg = explainer_eg(X_enc_eg)




### Subgroup Feature Importance (Exponentiated Gradient)

The following plots show subgroup feature importance and differences after mitigation.

Comparing these results with the baseline allows evaluation of whether fairness interventions alter model reasoning or simply adjust decision boundaries.

In [9]:
# ================================
# SUBGROUP SPLIT (EG)
# ================================

male_idx = A[A == "Male"].index
female_idx = A[A == "Female"].index

shap_male_eg = shap_vals_eg.values[male_idx]
shap_female_eg = shap_vals_eg.values[female_idx]

male_imp_eg = np.abs(shap_male_eg).mean(axis=0)
female_imp_eg = np.abs(shap_female_eg).mean(axis=0)

df_eg = pd.DataFrame({
    "feature": pre_eg.get_feature_names_out(),
    "Male": male_imp_eg,
    "Female": female_imp_eg,
})

df_eg["diff"] = df_eg["Male"] - df_eg["Female"]


# ================================
# PLOT 3 — EG FEATURE IMPORTANCE
# ================================

top_eg = df_eg.sort_values("Male", ascending=False).head(5)

x = np.arange(len(top_eg))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, top_eg["Male"], width, label="Male")
plt.bar(x + width/2, top_eg["Female"], width, label="Female")

plt.xticks(x, top_eg["feature"], rotation=45, ha="right")
plt.ylabel("Mean |SHAP|")
plt.title("Subgroup Feature Importance (Exponentiated Gradient – DP)")
plt.legend()

plt.tight_layout()
plt.savefig(out / "shap_subgroup_expgrad_dp.png", dpi=300)
plt.close()


# ================================
# PLOT 4 — EG DIFFERENCES
# ================================

top_diff_eg = df_eg.sort_values("diff", key=abs, ascending=False).head(5)

plt.figure(figsize=(7, 4))
plt.barh(top_diff_eg["feature"], top_diff_eg["diff"])
plt.axvline(0, color="black", linewidth=0.8)

plt.xlabel("Male − Female Mean |SHAP|")
plt.title("Subgroup SHAP Differences (Exponentiated Gradient – DP)")

plt.tight_layout()
plt.savefig(out / "shap_subgroup_diff_expgrad_dp.png", dpi=300)
plt.close()